# EDGY Exported-Models Inference Notebook

This notebook runs inference using artifacts in `exported_models` only.

It supports:
- Single WAV or batch WAV processing from `TestAudioFiles`
- ONNX encoder inference (FP32 default, optional INT8)
- Three-tier privacy processing (`LOW`, `MODERATE`, `HIGH`) inspired by DDF/replication Section 9
- Two decoder backends for `MODERATE`/`HIGH`:
  - ONNX WaveRNN-style autoregressive decoding (`wavernn_autoregressive`)
  - Griffin-Lim reconstruction fallback (`griffinlim_reconstruction`, non-ML decoder)

No checkpoints are loaded in this notebook.

In [ ]:
from pathlib import Path

# ===== User configuration =====
PROJECT_ROOT = Path('.').resolve()
MODEL_DIR = PROJECT_ROOT / 'exported_models'
INPUT_DIR = PROJECT_ROOT / 'TestAudioFiles'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'processed_audio'
ARTIFACT_DIR = OUTPUT_DIR / 'artifacts'

# Optional: set to a specific file path to process only one WAV
SINGLE_INPUT_FILE = None  # e.g. PROJECT_ROOT / 'TestAudioFiles' / 'example.wav'

# If input folder is empty, create a short synthetic WAV for smoke testing
GENERATE_DEMO_IF_EMPTY = True

# ONNX selection
USE_INT8_ENCODER = False
USE_INT8_DECODER = False

# Privacy tier selection (DDF-style behavior): LOW | MODERATE | HIGH
PRIVACY_TIER = 'HIGH'

# Optional direct override for decoding behavior (set None to use PRIVACY_TIER mapping)
# Valid values: 'raw_passthrough', 'griffinlim_reconstruction', 'wavernn_autoregressive'
DECODER_MODE = 'griffinlim_reconstruction'

# Griffin-Lim fallback controls (used when mode == 'griffinlim_reconstruction')
GRIFFINLIM_ITERATIONS = 32
GRIFFINLIM_MOMENTUM = 0.99

# MODERATE/HIGH speaker handling (used by WaveRNN ONNX decode only)
MODERATE_SPEAKER_ID = 0
HIGH_SYNTH_SPEAKER_ID = 1

# Autoregressive decoding controls (used by WaveRNN ONNX decode only)
WAVERNN_STRATEGY = 'argmax'  # 'argmax' or 'sample'
WAVERNN_TEMPERATURE = 1.0

# Runtime behavior
OVERWRITE_OUTPUTS = True
VERBOSE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Model dir:    {MODEL_DIR}')
print(f'Input dir:    {INPUT_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Privacy tier: {PRIVACY_TIER}')

Project root: C:\Users\CarstenZ\OneDrive - Imperial College London\Security Module Dev\EDGY
Model dir:    C:\Users\CarstenZ\OneDrive - Imperial College London\Security Module Dev\EDGY\exported_models
Input dir:    C:\Users\CarstenZ\OneDrive - Imperial College London\Security Module Dev\EDGY\TestAudioFiles
Output dir:   C:\Users\CarstenZ\OneDrive - Imperial College London\Security Module Dev\EDGY\output\processed_audio
Privacy tier: HIGH


In [55]:
import importlib
import subprocess
import sys

required_pkgs = [
    ("numpy", "numpy"),
    ("librosa", "librosa"),
    ("onnxruntime", "onnxruntime"),
    ("soundfile", "soundfile"),
    ("IPython", "ipython"),
]

missing = [pip_name for import_name, pip_name in required_pkgs if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

All required packages are already available.


In [56]:
import json
import time
import hashlib

import numpy as np
import librosa
import onnxruntime as ort
import soundfile as sf

from IPython.display import Audio, display

np.random.seed(42)

required = [
    MODEL_DIR / 'model_config.json',
    MODEL_DIR / 'edgy_encoder_fp32.onnx',
    MODEL_DIR / 'vq_codebook.npy',
    MODEL_DIR / 'speaker_embeddings.npy',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required exported model files:\n' + '\n'.join(missing))

with open(MODEL_DIR / 'model_config.json', 'r', encoding='utf-8') as f:
    model_config = json.load(f)

prep = model_config['preprocessing']
enc = model_config['encoder']
dec = model_config.get('decoder', {})
files_cfg = model_config.get('files', {})

SR = int(prep['sample_rate'])
N_FFT = int(prep['n_fft'])
N_MELS = int(prep['n_mels'])
HOP_LENGTH = int(prep['hop_length'])
WIN_LENGTH = int(prep['win_length'])
FMIN = float(prep['fmin'])
PREEMPH = float(prep['preemph'])
TOP_DB = float(prep['top_db'])
BITS = int(prep.get('bits', 8))

EMBEDDING_DIM = int(enc['embedding_dim'])
N_EMBEDDINGS = int(enc['n_embeddings'])
DOWNSAMPLING_FACTOR = int(enc.get('downsampling_factor', 1))

QUANTIZATION_CHANNELS = int(dec.get('quantization_channels', 2 ** BITS))
RNN_CHANNELS = int(dec.get('rnn_channels', 896))
DEC_COND_CHANNELS = int(dec.get('conditioning_channels', 256))

codebook = np.load(MODEL_DIR / 'vq_codebook.npy')
speaker_embeddings = np.load(MODEL_DIR / 'speaker_embeddings.npy')

if codebook.shape[1] != EMBEDDING_DIM:
    raise ValueError(f'Codebook dim mismatch: {codebook.shape[1]} != {EMBEDDING_DIM}')

# Encoder ONNX
encoder_onnx_path = MODEL_DIR / ('edgy_encoder_int8.onnx' if USE_INT8_ENCODER else 'edgy_encoder_fp32.onnx')
if USE_INT8_ENCODER and not encoder_onnx_path.exists():
    print('INT8 encoder ONNX not found, falling back to FP32 encoder ONNX')
    encoder_onnx_path = MODEL_DIR / 'edgy_encoder_fp32.onnx'

encoder_sess = ort.InferenceSession(str(encoder_onnx_path), providers=['CPUExecutionProvider'])
encoder_input_name = encoder_sess.get_inputs()[0].name
encoder_output_names = [o.name for o in encoder_sess.get_outputs()]

# WaveRNN decoder ONNX (conditioner + step)
cond_fp32_name = files_cfg.get('wavernn_conditioner_fp32', 'wavernn_conditioner_fp32.onnx')
step_fp32_name = files_cfg.get('wavernn_step_fp32', 'wavernn_step_fp32.onnx')
cond_int8_name = files_cfg.get('wavernn_conditioner_int8', 'wavernn_conditioner_int8.onnx')
step_int8_name = files_cfg.get('wavernn_step_int8', 'wavernn_step_int8.onnx')

cond_path = MODEL_DIR / (cond_int8_name if USE_INT8_DECODER else cond_fp32_name)
step_path = MODEL_DIR / (step_int8_name if USE_INT8_DECODER else step_fp32_name)
if USE_INT8_DECODER and (not cond_path.exists() or not step_path.exists()):
    print('INT8 decoder ONNX not found, falling back to FP32 decoder ONNX')
    cond_path = MODEL_DIR / cond_fp32_name
    step_path = MODEL_DIR / step_fp32_name

decoder_onnx_available = cond_path.exists() and step_path.exists()
if decoder_onnx_available:
    cond_sess = ort.InferenceSession(str(cond_path), providers=['CPUExecutionProvider'])
    step_sess = ort.InferenceSession(str(step_path), providers=['CPUExecutionProvider'])

    cond_input_names = [i.name for i in cond_sess.get_inputs()]
    cond_output_name = cond_sess.get_outputs()[0].name

    step_input_names = [i.name for i in step_sess.get_inputs()]
    step_output_names = [o.name for o in step_sess.get_outputs()]

    print(f'Loaded decoder conditioner ONNX: {cond_path.name}')
    print(f'Loaded decoder step ONNX:        {step_path.name}')
else:
    cond_sess = None
    step_sess = None
    cond_input_names = []
    cond_output_name = None
    step_input_names = []
    step_output_names = []
    print('⚠ WaveRNN decoder ONNX files not found in exported_models.')
    print('  Run replication notebook Section 12 export cell first.')

print(f'Loaded encoder ONNX: {encoder_onnx_path.name}')
print(f'Encoder input:       {encoder_input_name}')
print(f'Encoder outputs:     {encoder_output_names}')
print(f'Codebook:            {codebook.shape}')
print(f'Speakers:            {speaker_embeddings.shape}')
print(f'Downsample factor:   x{DOWNSAMPLING_FACTOR}')
print(f'Decoder available:   {decoder_onnx_available}')

Loaded decoder conditioner ONNX: wavernn_conditioner_fp32.onnx
Loaded decoder step ONNX:        wavernn_step_fp32.onnx
Loaded encoder ONNX: edgy_encoder_fp32.onnx
Encoder input:       mel_spectrogram
Encoder outputs:     ['vq_embedding', 'codebook_indices']
Codebook:            (512, 64)
Speakers:            (40, 64)
Downsample factor:   x2
Decoder available:   True


In [57]:
def _stable_seed_from_array(arr: np.ndarray) -> int:
    h = hashlib.sha256(arr.tobytes()).hexdigest()
    return int(h[:8], 16)

def mulaw_decode_np(y: np.ndarray, quantization_channels: int) -> np.ndarray:
    """Inverse mu-law for y in [-1, 1]."""
    mu = float(quantization_channels - 1)
    x = np.sign(y) / mu * ((1.0 + mu) ** np.abs(y) - 1.0)
    return x.astype(np.float32)

def _pick_name(candidates: list[str], index: int, fallback: str) -> str:
    if fallback in candidates:
        return fallback
    if index < len(candidates):
        return candidates[index]
    raise ValueError(f'Could not resolve input/output name: {fallback}')

def _softmax_np(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    t = max(float(temperature), 1e-6)
    z = logits / t
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

print('Utility helpers ready.')

Utility helpers ready.


In [58]:
def preprocess_wav_to_logmel(wav_path: Path) -> tuple[np.ndarray, np.ndarray]:
    wav, _ = librosa.load(str(wav_path), sr=SR, mono=True)
    wav = wav.astype(np.float32)

    peak = np.max(np.abs(wav))
    if peak > 0:
        wav = wav / peak * 0.999

    # Match EDGY-style preemphasis used in training/export notebooks.
    wav_pre = librosa.effects.preemphasis(wav, coef=PREEMPH)

    mel = librosa.feature.melspectrogram(
        y=wav_pre,
        sr=SR,
        n_fft=N_FFT,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        fmin=FMIN,
        power=1.0,
    )

    logmel_db = librosa.amplitude_to_db(mel, top_db=TOP_DB)
    logmel_norm = (logmel_db / TOP_DB) + 1.0
    return wav, logmel_norm.astype(np.float32)

def encode_logmel_onnx(logmel_norm: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    model_in = logmel_norm[np.newaxis, :, :].astype(np.float32)
    outputs = encoder_sess.run(None, {encoder_input_name: model_in})

    # Expected order from export: [vq_embedding, codebook_indices]
    vq_embedding = outputs[0]
    indices = outputs[1]

    if vq_embedding.shape[-1] != EMBEDDING_DIM:
        raise ValueError(f'Unexpected VQ dim: {vq_embedding.shape[-1]} != {EMBEDDING_DIM}')

    if indices.size > 0:
        idx_min = int(np.min(indices))
        idx_max = int(np.max(indices))
        if idx_min < 0 or idx_max >= N_EMBEDDINGS:
            raise ValueError(f'Codebook indices out of range: min={idx_min}, max={idx_max}')

    return vq_embedding.astype(np.float32), indices.astype(np.int64)

def reconstruct_wav_griffinlim(logmel_norm: np.ndarray) -> tuple[np.ndarray, dict[str, int | float]]:
    # Convert normalized log-mel back to amplitude mel before Griffin-Lim inversion.
    logmel_db = (logmel_norm.astype(np.float32) - 1.0) * float(TOP_DB)
    mel_amp = librosa.db_to_amplitude(logmel_db)

    kwargs = dict(
        M=mel_amp,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        fmin=FMIN,
        power=1.0,
        n_iter=int(GRIFFINLIM_ITERATIONS),
    )

    # Compatibility for librosa versions where mel_to_audio does not expose momentum.
    try:
        wav = librosa.feature.inverse.mel_to_audio(
            **kwargs,
            momentum=float(GRIFFINLIM_MOMENTUM),
        ).astype(np.float32)
    except TypeError:
        wav = librosa.feature.inverse.mel_to_audio(**kwargs).astype(np.float32)

    # Approximate inversion of analysis preemphasis.
    wav = librosa.effects.deemphasis(wav, coef=PREEMPH).astype(np.float32)

    peak = np.max(np.abs(wav))
    if peak > 0:
        wav = wav / peak * 0.98

    dbg = {
        'griffinlim_iterations': int(GRIFFINLIM_ITERATIONS),
        'griffinlim_momentum': float(GRIFFINLIM_MOMENTUM),
        'mel_frames': int(logmel_norm.shape[1]),
    }
    return wav, dbg

def run_wavernn_onnx_decoder(
    vq_embedding: np.ndarray,
    speaker_id: int,
    strategy: str = 'argmax',
    temperature: float = 1.0,
    ) -> tuple[np.ndarray, dict[str, int]]:
    if not decoder_onnx_available:
        raise RuntimeError('WaveRNN decoder ONNX is unavailable. Export decoder ONNX first.')

    cond_z_name = _pick_name(cond_input_names, 0, 'z_q')
    cond_spk_name = _pick_name(cond_input_names, 1, 'speaker_id')

    conditioning = cond_sess.run(
        [cond_output_name],
        {
            cond_z_name: vq_embedding.astype(np.float32),
            cond_spk_name: np.array([int(speaker_id)], dtype=np.int64),
        },
    )[0]  # [1, T_audio, C]

    step_prev_mu_name = _pick_name(step_input_names, 0, 'prev_mu')
    step_cond_name = _pick_name(step_input_names, 1, 'cond_t')
    step_h_name = _pick_name(step_input_names, 2, 'h_prev')

    logits_name = _pick_name(step_output_names, 0, 'logits')
    h_next_name = _pick_name(step_output_names, 1, 'h_next')

    h = np.zeros((1, RNN_CHANNELS), dtype=np.float32)
    prev_mu = np.array([QUANTIZATION_CHANNELS // 2], dtype=np.int64)

    mu_indices = []
    n_steps = int(conditioning.shape[1])

    for t in range(n_steps):
        cond_t = conditioning[:, t, :].astype(np.float32)
        logits, h = step_sess.run(
            [logits_name, h_next_name],
            {
                step_prev_mu_name: prev_mu,
                step_cond_name: cond_t,
                step_h_name: h,
            },
        )

        if strategy == 'argmax':
            next_mu = np.argmax(logits, axis=1).astype(np.int64)
        elif strategy == 'sample':
            probs = _softmax_np(logits.astype(np.float64), temperature=temperature)
            next_mu = np.array([np.random.choice(QUANTIZATION_CHANNELS, p=probs[0])], dtype=np.int64)
        else:
            raise ValueError(f'Unsupported WAVERNN_STRATEGY: {strategy}')

        prev_mu = next_mu
        mu_indices.append(int(next_mu[0]))

    mu_indices_np = np.asarray(mu_indices, dtype=np.float32)
    mu_codes = (2.0 * mu_indices_np / float(QUANTIZATION_CHANNELS - 1)) - 1.0
    wav = mulaw_decode_np(mu_codes, QUANTIZATION_CHANNELS)

    peak = np.max(np.abs(wav))
    if peak > 0:
        wav = wav / peak * 0.98

    dbg = {
        'conditioning_frames': int(conditioning.shape[1]),
        'decoded_steps': int(len(mu_indices)),
        'speaker_id': int(speaker_id),
    }
    return wav.astype(np.float32), dbg

def resolve_tier_and_mode(privacy_tier: str, decoder_mode_override=None) -> tuple[str, str]:
    tier = str(privacy_tier).strip().upper()
    if tier not in {'LOW', 'MODERATE', 'HIGH'}:
        raise ValueError(f'Invalid PRIVACY_TIER: {privacy_tier}. Use LOW, MODERATE, or HIGH.')

    if decoder_mode_override is not None:
        mode = decoder_mode_override
    else:
        # Temporary default: use Griffin-Lim for MODERATE/HIGH while exported WaveRNN is unstable.
        # Swap back by setting DECODER_MODE = 'wavernn_autoregressive'.
        mode = {
            'LOW': 'raw_passthrough',
            'MODERATE': 'griffinlim_reconstruction',
            'HIGH': 'griffinlim_reconstruction',
        }[tier]

    if mode not in {'raw_passthrough', 'griffinlim_reconstruction', 'wavernn_autoregressive'}:
        raise ValueError(f'Invalid decode mode: {mode}')

    return tier, mode

def decode_by_tier(
    wav_in: np.ndarray,
    logmel_in: np.ndarray,
    vq_embedding: np.ndarray,
    privacy_tier: str,
    decoder_mode_override=None,
    strategy: str = 'argmax',
    temperature: float = 1.0,
    ) -> tuple[np.ndarray, str, dict[str, int | float] | None]:
    tier, effective_mode = resolve_tier_and_mode(privacy_tier, decoder_mode_override)

    if effective_mode == 'raw_passthrough':
        wav_out = wav_in.copy()
        peak = np.max(np.abs(wav_out))
        if peak > 0:
            wav_out = wav_out / peak * 0.98
        return wav_out.astype(np.float32), effective_mode, None

    if effective_mode == 'griffinlim_reconstruction':
        wav_out, dbg = reconstruct_wav_griffinlim(logmel_in)
        dbg = dict(dbg)
        dbg['privacy_tier'] = tier
        return wav_out.astype(np.float32), effective_mode, dbg

    if not decoder_onnx_available:
        raise RuntimeError('Decoder ONNX is required when mode is wavernn_autoregressive.')

    # MODERATE preserves identity more by using explicit speaker embedding id.
    # HIGH uses a synthetic/alternate speaker id to reduce identity carry-over.
    if tier == 'MODERATE':
        speaker_id = int(MODERATE_SPEAKER_ID)
    else:
        speaker_id = int(HIGH_SYNTH_SPEAKER_ID)

    n_speakers = int(speaker_embeddings.shape[0])
    speaker_id = max(0, min(speaker_id, n_speakers - 1))

    wav_out, dbg = run_wavernn_onnx_decoder(
        vq_embedding=vq_embedding,
        speaker_id=speaker_id,
        strategy=strategy,
        temperature=temperature,
    )
    return wav_out, effective_mode, dbg

def collect_input_files(input_dir: Path, single_file=None):
    if single_file is not None:
        p = Path(single_file)
        if not p.exists():
            raise FileNotFoundError(f'SINGLE_INPUT_FILE does not exist: {p}')
        return [p]

    if not input_dir.exists():
        raise FileNotFoundError(f'Input directory not found: {input_dir}')

    files = sorted(input_dir.rglob('*.wav'))
    return files

In [59]:
INPUT_DIR.mkdir(parents=True, exist_ok=True)

if GENERATE_DEMO_IF_EMPTY and SINGLE_INPUT_FILE is None:
    existing_wavs = sorted(INPUT_DIR.rglob('*.wav'))
    if not existing_wavs:
        demo_path = INPUT_DIR / 'demo_sine_220hz.wav'
        dur_sec = 2.0
        t = np.linspace(0, dur_sec, int(SR * dur_sec), endpoint=False, dtype=np.float32)
        wav_demo = 0.2 * np.sin(2 * np.pi * 220.0 * t)
        sf.write(str(demo_path), wav_demo, SR)
        print(f'Generated demo input WAV: {demo_path}')
    else:
        print(f'Input WAVs already present: {len(existing_wavs)}')

Input WAVs already present: 1


In [60]:
input_files = collect_input_files(INPUT_DIR, SINGLE_INPUT_FILE)
print(f'Found {len(input_files)} wav file(s)')
for p in input_files[:10]:
    print(' -', p)
if len(input_files) > 10:
    print(' ...')

Found 1 wav file(s)
 - C:\Users\CarstenZ\OneDrive - Imperial College London\Security Module Dev\EDGY\TestAudioFiles\Test.wav


In [61]:
results = []
tier_selected, _ = resolve_tier_and_mode(PRIVACY_TIER, DECODER_MODE)

for wav_path in input_files:
    t0 = time.perf_counter()
    stem = wav_path.stem

    tier_tag = tier_selected.lower()
    out_wav = OUTPUT_DIR / f'{stem}_edgy_{tier_tag}.wav'
    out_vq = ARTIFACT_DIR / f'{stem}_{tier_tag}_vq.npy'
    out_idx = ARTIFACT_DIR / f'{stem}_{tier_tag}_indices.npy'
    out_meta = ARTIFACT_DIR / f'{stem}_{tier_tag}_meta.json'

    if (not OVERWRITE_OUTPUTS) and out_wav.exists():
        results.append({
            'file': str(wav_path),
            'status': 'skipped_exists',
            'output_wav': str(out_wav),
        })
        continue

    try:
        wav_in, logmel_in = preprocess_wav_to_logmel(wav_path)
        vq, indices = encode_logmel_onnx(logmel_in)

        wav_out, effective_mode, decode_dbg = decode_by_tier(
            wav_in=wav_in,
            logmel_in=logmel_in,
            vq_embedding=vq,
            privacy_tier=PRIVACY_TIER,
            decoder_mode_override=DECODER_MODE,
            strategy=WAVERNN_STRATEGY,
            temperature=WAVERNN_TEMPERATURE,
        )

        sf.write(str(out_wav), wav_out, SR)

        np.save(out_vq, vq)
        np.save(out_idx, indices)

        dt = time.perf_counter() - t0
        in_dur = float(len(wav_in) / SR)
        out_dur = float(len(wav_out) / SR)
        duration_ratio = out_dur / max(in_dur, 1e-8)

        meta = {
            'input_file': str(wav_path),
            'output_wav': str(out_wav),
            'vq_file': str(out_vq),
            'indices_file': str(out_idx),
            'privacy_tier': tier_selected,
            'effective_decode_mode': effective_mode,
            'onnx_encoder_model': encoder_onnx_path.name,
            'onnx_decoder_conditioner': cond_path.name if decoder_onnx_available else None,
            'onnx_decoder_step': step_path.name if decoder_onnx_available else None,
            'sample_rate': SR,
            'quantization_channels': QUANTIZATION_CHANNELS,
            'wavernn_strategy': WAVERNN_STRATEGY,
            'wavernn_temperature': WAVERNN_TEMPERATURE,
            'griffinlim_iterations': int(GRIFFINLIM_ITERATIONS),
            'griffinlim_momentum': float(GRIFFINLIM_MOMENTUM),
            'downsampling_factor': int(DOWNSAMPLING_FACTOR),
            'input_duration_sec': round(in_dur, 4),
            'output_duration_sec': round(out_dur, 4),
            'duration_ratio': round(duration_ratio, 4),
            'input_mel_frames': int(logmel_in.shape[1]),
            'encoded_frames': int(vq.shape[1]),
            'decode_debug': decode_dbg,
            'index_min': int(np.min(indices)) if indices.size else None,
            'index_max': int(np.max(indices)) if indices.size else None,
            'elapsed_sec': round(dt, 4),
            'status': 'ok',
        }

        with open(out_meta, 'w', encoding='utf-8') as f:
            json.dump(meta, f, indent=2)

        results.append(meta)
        if VERBOSE:
            print(
                f"OK  | {wav_path.name} -> {out_wav.name} | "
                f"tier={tier_selected} mode={effective_mode} | "
                f"{meta['input_duration_sec']:.2f}s -> {meta['output_duration_sec']:.2f}s "
                f"(x{meta['duration_ratio']:.3f}) | {dt:.2f}s"
            )

    except Exception as e:
        dt = time.perf_counter() - t0
        err = {
            'input_file': str(wav_path),
            'status': 'failed',
            'error': str(e),
            'elapsed_sec': round(dt, 4),
        }
        results.append(err)
        print(f'ERR | {wav_path.name} | {e}')

OK  | Test.wav -> Test_edgy_high.wav | tier=HIGH mode=wavernn_autoregressive | 11.83s -> 11.82s (x0.999) | 20.04s


In [62]:
ok = [r for r in results if r.get('status') == 'ok']
failed = [r for r in results if r.get('status') == 'failed']
skipped = [r for r in results if r.get('status') == 'skipped_exists']

tier_selected, mode_selected = resolve_tier_and_mode(PRIVACY_TIER, DECODER_MODE)

print('=' * 72)
print(f'Total files : {len(results)}')
print(f'Succeeded   : {len(ok)}')
print(f'Failed      : {len(failed)}')
print(f'Skipped     : {len(skipped)}')
print(f'Model used  : {encoder_onnx_path.name}')
print(f'Tier        : {tier_selected}')
print(f'Mode        : {mode_selected}')
print(f'Output dir  : {OUTPUT_DIR}')
print('=' * 72)

if failed:
    print('Failures:')
    for f in failed:
        print(' -', Path(f['input_file']).name, '|', f['error'])

if ok:
    first = Path(ok[0]['output_wav'])
    print(f'\nPreview audio: {first.name}')
    display(Audio(filename=str(first), rate=SR))

Total files : 1
Succeeded   : 1
Failed      : 0
Skipped     : 0
Model used  : edgy_encoder_fp32.onnx
Tier        : HIGH
Mode        : wavernn_autoregressive
Output dir  : C:\Users\CarstenZ\OneDrive - Imperial College London\Security Module Dev\EDGY\output\processed_audio

Preview audio: Test_edgy_high.wav


In [63]:
import numpy as np
from pathlib import Path

print('=== Compact Debug Summary ===')
print(f"results entries: {len(results) if 'results' in globals() else 'N/A'}")

if 'results' not in globals() or not results:
    print('No results in memory. Re-run inference cell first.')
else:
    ok = [r for r in results if r.get('status') == 'ok']
    failed = [r for r in results if r.get('status') == 'failed']
    skipped = [r for r in results if r.get('status') == 'skipped_exists']

    print(f"ok={len(ok)} failed={len(failed)} skipped={len(skipped)}")
    mode_counts = {}
    for r in ok:
        m = r.get('effective_decode_mode', 'unknown')
        mode_counts[m] = mode_counts.get(m, 0) + 1
    print('mode counts:', mode_counts)

    if failed:
        print('\nTop failures:')
        for f in failed[:5]:
            print('-', Path(f.get('input_file', 'unknown')).name, '|', f.get('error', 'no error text'))

    # Inspect a few produced WAVs for silence/clipping clues.
    print('\nAudio health checks (first 5 ok files):')
    for r in ok[:5]:
        p = Path(r['output_wav'])
        if not p.exists():
            print('-', p.name, '| missing output file')
            continue
        import soundfile as sf
        wav, sr = sf.read(str(p), dtype='float32')
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        peak = float(np.max(np.abs(wav))) if wav.size else 0.0
        rms = float(np.sqrt(np.mean(np.square(wav)))) if wav.size else 0.0
        near_silence = rms < 1e-3
        likely_clipped = peak >= 0.98
        print(f"- {p.name}: dur={len(wav)/sr:.2f}s peak={peak:.4f} rms={rms:.6f} silent={near_silence} clipped={likely_clipped}")

=== Compact Debug Summary ===
results entries: 1
ok=1 failed=0 skipped=0
mode counts: {'wavernn_autoregressive': 1}

Audio health checks (first 5 ok files):
- Test_edgy_high.wav: dur=11.82s peak=0.9800 rms=0.979980 silent=False clipped=False
